# Module 2.7 — Distillation: Shrink It Further (Optional)
**DeskMate SLM Curriculum · Phase 2 · Notebook 14**

Run on **CPU** (no GPU needed — models are small, gold set is small).

By the end you will have:
- A 6-layer DistilBERT student that mimics the 12-layer MiniLM teacher
- A side-by-side quality + latency benchmark
- A written deployment recommendation

Read `2.7_distillation.md` for the full theory.

---

## Step 0 — Install & Seed

In [ ]:
%%capture
!pip install -q transformers==4.44.0 datasets==2.21.0 torch==2.3.1 scikit-learn==1.5.1

In [ ]:
import random, json, pathlib, time
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset, DatasetDict, load_from_disk
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments, Trainer,
)
from sklearn.metrics import f1_score, accuracy_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device}')

## Step 1 — Runtime & Paths

In [ ]:
try:
    import google.colab; RUNTIME = 'colab'
    PROJECT_ROOT = pathlib.Path('/content/slm')
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = 'kaggle'
        PROJECT_ROOT = pathlib.Path('/kaggle/working/slm')
    except ImportError:
        RUNTIME = 'local'
        PROJECT_ROOT = pathlib.Path('.').resolve()

DATA_PROC  = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Runtime : {RUNTIME}')

## Step 2 — Label Maps

In [ ]:
maps_path = DATA_PROC / 'label_maps.json'
if maps_path.exists():
    lm = json.loads(maps_path.read_text())
    INTENT2ID = lm['INTENT2ID']
    ID2INTENT = {int(k): v for k, v in lm['ID2INTENT'].items()}
else:
    INTENTS   = ['account_access','account_settings','billing_dispute','billing_inquiry',
                 'cancellation','data_privacy','feature_request','onboarding',
                 'outage_report','payment_failure','performance_issue','refund_request',
                 'technical_bug','usage_question','out_of_scope']
    INTENT2ID = {i: n for n, i in enumerate(INTENTS)}
    ID2INTENT = {n: i for n, i in enumerate(INTENTS)}

N_INTENTS = len(INTENT2ID)
print(f'Intent classes: {N_INTENTS}')

## Step 3 — Load Tokenizer & Teacher Model

In [ ]:
TEACHER_NAME = 'microsoft/MiniLM-L12-H384-uncased'
tokenizer    = AutoTokenizer.from_pretrained(TEACHER_NAME)
collator     = DataCollatorWithPadding(tokenizer=tokenizer)

teacher_path = MODELS_DIR / 'intent_classifier'
if teacher_path.exists():
    teacher = AutoModelForSequenceClassification.from_pretrained(str(teacher_path))
    print(f'Loaded fine-tuned teacher from {teacher_path}')
else:
    print('WARNING: fine-tuned teacher not found — using pretrained weights')
    print('         Train Module 2.4 first for meaningful distillation.')
    teacher = AutoModelForSequenceClassification.from_pretrained(
        TEACHER_NAME, num_labels=N_INTENTS, id2label=ID2INTENT, label2id=INTENT2ID)

teacher.eval()
n_params = sum(p.numel() for p in teacher.parameters())
n_layers = teacher.config.num_hidden_layers
print(f'Teacher: {n_params/1e6:.1f}M params, {n_layers} layers')

## Step 4 — Load Training Dataset

In [ ]:
tok_path = DATA_PROC / 'tokenized'
if tok_path.exists():
    tokenized = load_from_disk(str(tok_path))
    print(f'Loaded tokenized dataset: {list(tokenized.keys())}')
else:
    print('Tokenized dataset not found — building placeholder from JSONL')
    def load_raw(name):
        p = DATA_PROC / f'{name}.jsonl'
        if not p.exists():
            rows = [
                {'text': 'I cannot log in.', 'intent': 'account_access', 'priority': 'medium'},
                {'text': 'Charged twice.', 'intent': 'billing_dispute', 'priority': 'high'},
                {'text': 'How do I export?', 'intent': 'usage_question', 'priority': 'low'},
            ] * 20
            p.parent.mkdir(parents=True, exist_ok=True)
            p.write_text('\n'.join(json.dumps(r) for r in rows))
        return Dataset.from_pandas(pd.read_json(p, lines=True))

    aug = DATA_PROC / 'train_augmented.jsonl'
    raw = DatasetDict({
        'train': load_raw('train_augmented' if aug.exists() else 'train'),
        'val':   load_raw('val'),
        'test':  load_raw('test'),
    })
    def tok_fn(ex):
        out = tokenizer(ex['text'], truncation=True, max_length=128, padding=False)
        out['intent_label'] = [INTENT2ID.get(i, 0) for i in ex['intent']]
        return out
    drop_cols = [c for c in raw['train'].column_names
                 if c not in ('input_ids','attention_mask','intent_label')]
    tokenized = raw.map(tok_fn, batched=True, remove_columns=drop_cols)

# Rename intent_label -> labels
def rename_labels(ex):
    ex['labels'] = ex['intent_label']
    return ex

train_ds = tokenized['train'].map(rename_labels, batched=True)
val_ds   = tokenized['val'].map(rename_labels, batched=True)
test_ds  = tokenized['test'].map(rename_labels, batched=True)

for name, ds in [('train', train_ds), ('val', val_ds), ('test', test_ds)]:
    print(f'  {name}: {len(ds):>6} examples')

train_ds.set_format('torch', columns=['input_ids','attention_mask','labels'])
val_ds.set_format('torch',   columns=['input_ids','attention_mask','labels'])
test_ds.set_format('torch',  columns=['input_ids','attention_mask','labels'])

## Step 5 — Generate Soft Labels (Teacher Logits)

Run the frozen teacher over the full training set once and cache the logits.
These are used in place of one-hot labels during student training.

In [ ]:
soft_path = DATA_PROC / 'teacher_logits_train.pt'

if soft_path.exists():
    teacher_logits_all = torch.load(soft_path)
    print(f'Loaded cached teacher logits: {tuple(teacher_logits_all.shape)}')
else:
    print('Generating teacher soft labels ...')
    loader = DataLoader(train_ds, batch_size=64, collate_fn=collator, shuffle=False)
    all_logits = []
    teacher.eval()
    with torch.no_grad():
        for batch in loader:
            out = teacher(input_ids=batch['input_ids'],
                          attention_mask=batch['attention_mask'])
            all_logits.append(out.logits.cpu())
    teacher_logits_all = torch.cat(all_logits, dim=0)
    torch.save(teacher_logits_all, soft_path)
    print(f'Teacher logits saved: {tuple(teacher_logits_all.shape)}')

# Show soft label for first example
soft = teacher_logits_all[0].softmax(dim=-1)
print('\nTeacher soft label for first training example:')
top5 = soft.topk(5)
for val, idx in zip(top5.values, top5.indices):
    print(f'  {ID2INTENT[idx.item()]:<25}: {val.item():.4f}')

## Step 6 — Distillation Loss Function

Combines cross-entropy on hard labels with KL divergence on temperature-scaled soft labels.

In [ ]:
def distillation_loss(student_logits, teacher_logits, hard_labels, T=4.0, alpha=0.4):
    ce_loss     = F.cross_entropy(student_logits, hard_labels)
    soft_teacher = F.log_softmax(teacher_logits / T, dim=-1)
    soft_student = F.log_softmax(student_logits / T, dim=-1)
    kd_loss     = F.kl_div(soft_student, soft_teacher.exp(),
                            reduction='batchmean') * (T ** 2)
    return alpha * ce_loss + (1 - alpha) * kd_loss

# Unit test
torch.manual_seed(0)
stu  = torch.randn(4, N_INTENTS)
tch  = torch.randn(4, N_INTENTS)
lbls = torch.randint(0, N_INTENTS, (4,))
loss = distillation_loss(stu, tch, lbls)
assert loss.item() > 0
print(f'distillation_loss unit test passed: {loss.item():.4f}')

# Visualise temperature effect on soft labels
sample_logits = teacher_logits_all[0]
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, T in zip(axes, [1.0, 4.0, 8.0]):
    soft = (sample_logits / T).softmax(dim=-1).numpy()
    ax.bar(range(N_INTENTS), soft, color='#4C72B0')
    ax.set_title(f'T = {T}')
    ax.set_xlabel('Class'); ax.set_ylabel('Prob')
    ax.set_xticks([])
plt.suptitle('Effect of temperature on teacher soft labels', y=1.02)
plt.tight_layout(); plt.show()
print('Higher T → flatter distribution → more inter-class information for student.')

## Step 7 — DistillationTrainer

We subclass `Trainer` and override `compute_loss` to inject teacher logits
for the current batch using a step counter.

In [ ]:
class DistillationTrainer(Trainer):
    def __init__(self, teacher_logits_all, T=4.0, alpha=0.4, **kwargs):
        super().__init__(**kwargs)
        self.teacher_logits_all = teacher_logits_all
        self.T     = T
        self.alpha = alpha
        self._example_idx = 0

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        out    = model(**inputs)
        bs     = labels.shape[0]
        start  = self._example_idx % len(self.teacher_logits_all)
        end    = min(start + bs, len(self.teacher_logits_all))
        t_logits = self.teacher_logits_all[start:end].to(out.logits.device)
        # Pad if batch wraps around end of dataset
        if t_logits.shape[0] < bs:
            t_logits = self.teacher_logits_all[:bs].to(out.logits.device)
        self._example_idx += bs
        loss = distillation_loss(out.logits, t_logits, labels, self.T, self.alpha)
        return (loss, out) if return_outputs else loss

print('DistillationTrainer defined.')

## Step 8 — Load Student Model

DistilBERT-base: 6 layers, d_model=768, 66M params.
Despite more parameters than MiniLM, it is ~40% faster due to half the depth.

In [ ]:
STUDENT_NAME = 'distilbert-base-uncased'
student = AutoModelForSequenceClassification.from_pretrained(
    STUDENT_NAME, num_labels=N_INTENTS, id2label=ID2INTENT, label2id=INTENT2ID)

s_params = sum(p.numel() for p in student.parameters())
s_layers = student.config.num_hidden_layers
t_params = sum(p.numel() for p in teacher.parameters())
t_layers = teacher.config.num_hidden_layers

print(f'Teacher: {t_params/1e6:.1f}M params, {t_layers} layers')
print(f'Student: {s_params/1e6:.1f}M params, {s_layers} layers')
print(f'Note: student has more params but fewer layers -> lower latency on CPU')

## Step 9 — Train Student with Distillation

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        'accuracy': float(accuracy_score(labels, preds)),
        'f1':       float(f1_score(labels, preds, average='macro', zero_division=0)),
    }

student_args = TrainingArguments(
    output_dir                  = str(MODELS_DIR / 'intent_classifier_distilled'),
    num_train_epochs            = 5,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    learning_rate               = 3e-5,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    greater_is_better           = True,
    fp16                        = (device == 'cuda'),
    seed                        = SEED,
    report_to                   = 'none',
)

dist_trainer = DistillationTrainer(
    teacher_logits_all = teacher_logits_all,
    T                  = 4.0,
    alpha              = 0.4,
    model              = student,
    args               = student_args,
    train_dataset      = train_ds,
    eval_dataset       = val_ds,
    data_collator      = collator,
    compute_metrics    = compute_metrics,
    tokenizer          = tokenizer,
)

print('Starting distillation training (5 epochs) ...')
dist_trainer.train()
print('Done.')

In [ ]:
# Plot val F1 per epoch for both teacher (static) and student (improving)
eval_logs  = [x for x in dist_trainer.state.log_history if 'eval_f1' in x]
student_f1 = [x['eval_f1'] for x in eval_logs]

teacher_preds = dist_trainer.predict(val_ds)
teacher_f1_val = f1_score(
    teacher_preds.label_ids,
    teacher_preds.predictions.argmax(axis=-1),
    average='macro', zero_division=0)

fig, ax = plt.subplots(figsize=(8, 4))
epochs = list(range(1, len(student_f1) + 1))
ax.plot(epochs, student_f1, 'o-', color='#4C72B0', label='Student (distilled)')
ax.axhline(teacher_f1_val, color='#DD8452', linestyle='--', label='Teacher (val F1)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val macro-F1')
ax.set_title('Distillation: Student vs Teacher')
ax.set_ylim(0, 1); ax.legend()
plt.tight_layout()
plt.savefig(str(MODELS_DIR / 'distillation_training.png'), bbox_inches='tight')
plt.show()

## Step 10 — Quality Benchmark on Test Set

In [ ]:
def evaluate_model(model, dataset, collator, label_name='test'):
    loader = DataLoader(dataset, batch_size=64, collate_fn=collator, shuffle=False)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            out   = model(input_ids=batch['input_ids'],
                          attention_mask=batch['attention_mask'])
            preds = out.logits.argmax(dim=-1)
            all_preds.extend(preds.tolist())
            all_labels.extend(batch['labels'].tolist())
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return acc, f1

teacher_acc, teacher_f1 = evaluate_model(teacher, test_ds, collator)
student_acc, student_f1 = evaluate_model(dist_trainer.model, test_ds, collator)

print('Quality on test set:')
print(f'  Teacher (MiniLM-12L): accuracy={teacher_acc*100:.1f}%  macro-F1={teacher_f1*100:.1f}%')
print(f'  Student (DistilBERT): accuracy={student_acc*100:.1f}%  macro-F1={student_f1*100:.1f}%')
print(f'  F1 gap: {(teacher_f1-student_f1)*100:.1f}pp')

## Step 11 — Latency Benchmark

In [ ]:
def measure_latency(model, tokenizer, texts, n_warmup=5, n_measure=100):
    model.eval()
    # Warmup
    for txt in texts[:n_warmup]:
        enc = tokenizer(txt, return_tensors='pt', truncation=True, max_length=128)
        with torch.no_grad():
            _ = model(**enc)
    # Measure
    times = []
    for txt in texts[:n_measure]:
        enc = tokenizer(txt, return_tensors='pt', truncation=True, max_length=128)
        t0 = time.perf_counter()
        with torch.no_grad():
            _ = model(**enc)
        times.append((time.perf_counter() - t0) * 1000)
    return np.mean(times), np.percentile(times, 95)

sample_texts = [
    'I cannot log in to my account, the password reset email never arrived.',
    'I was charged twice for the Pro plan last month, please refund the duplicate.',
    'The service has been down for 2 hours and our entire team is blocked.',
    'How do I set up the Slack integration for outbound notifications?',
    'Export to CSV throws a 500 error on Firefox and Chrome.',
] * 20

t_mean, t_p95 = measure_latency(teacher, tokenizer, sample_texts)
s_mean, s_p95 = measure_latency(dist_trainer.model, tokenizer, sample_texts)

print(f'Latency (single example, CPU, {device}):')
print(f'  Teacher (MiniLM-12L): mean={t_mean:.1f}ms  p95={t_p95:.1f}ms')
print(f'  Student (DistilBERT): mean={s_mean:.1f}ms  p95={s_p95:.1f}ms')
print(f'  Speedup: {t_mean/s_mean:.2f}x')

In [ ]:
# Side-by-side bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

models = ['Teacher\n(MiniLM-12L)', 'Student\n(DistilBERT-6L)']
ax1.bar(models, [teacher_f1*100, student_f1*100], color=['#4C72B0','#DD8452'])
ax1.set_ylabel('Macro-F1 (%)')
ax1.set_title('Quality (test set)')
ax1.set_ylim(0, 100)
for i, v in enumerate([teacher_f1*100, student_f1*100]):
    ax1.text(i, v+0.5, f'{v:.1f}%', ha='center', fontsize=11)

ax2.bar(models, [t_mean, s_mean], color=['#4C72B0','#DD8452'])
ax2.set_ylabel('Mean latency (ms)')
ax2.set_title('Latency (CPU, single example)')
for i, v in enumerate([t_mean, s_mean]):
    ax2.text(i, v+0.1, f'{v:.1f}ms', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig(str(MODELS_DIR / 'distillation_benchmark.png'), bbox_inches='tight')
plt.show()

## Step 12 — Deployment Recommendation

In [ ]:
f1_gap   = (teacher_f1 - student_f1) * 100
speedup  = t_mean / s_mean

print('=== Deployment Recommendation ===')
print()
if f1_gap < 2.0 and speedup > 1.3:
    verdict = 'DEPLOY STUDENT'
    reason  = (f'F1 gap is only {f1_gap:.1f}pp while latency improves {speedup:.2f}x. '
               f'Quality loss is acceptable for the latency benefit.')
elif f1_gap >= 2.0 and speedup > 2.0:
    verdict = 'DEPLOY STUDENT (with monitoring)'
    reason  = (f'F1 gap of {f1_gap:.1f}pp is non-trivial but {speedup:.2f}x speedup is significant. '
               f'Deploy student with A/B test and monitor gold-set F1 weekly.')
else:
    verdict = 'DEPLOY TEACHER'
    reason  = (f'F1 gap of {f1_gap:.1f}pp outweighs the {speedup:.2f}x latency gain. '
               f'At DeskMate current scale, teacher quality is worth the compute cost.')

print(f'Verdict : {verdict}')
print(f'Reason  : {reason}')
print()
print(f'Teacher: macro-F1={teacher_f1*100:.1f}%  latency={t_mean:.1f}ms')
print(f'Student: macro-F1={student_f1*100:.1f}%  latency={s_mean:.1f}ms')

## Step 13 — Save Student Model

In [ ]:
save_path = MODELS_DIR / 'intent_classifier_distilled'
dist_trainer.save_model(str(save_path))
tokenizer.save_pretrained(str(save_path))
print(f'Student model saved: {save_path}')

## Checkpoint

> **What does the student learn from soft labels that hard labels don't teach?**

Write your answer below. A strong answer covers:
1. What a hard label's probability distribution looks like.
2. What the teacher's soft label distribution reveals.
3. How temperature amplifies the inter-class signal.
4. Why this acts as regularisation.

In [ ]:
answer = """
[Write your answer here]
"""
print(answer)

## Deliverable Summary

| Artifact | Location |
|---|---|
| Distilled student model | `models/intent_classifier_distilled/` |
| Benchmark chart | `models/distillation_benchmark.png` |
| Training curve | `models/distillation_training.png` |

**Phase 2 complete (including optional 2.7).**

**Next: Phase 3** — The decoder SLM.
Module 3.0 establishes prompting baselines before fine-tuning.
Module 3.1 adapts Qwen2.5-1.5B with LoRA for support reply generation.
Module 3.2 handles structured / constrained decoding output.